# Librerías

In [1]:
import hashlib
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.validation import make_valid


def localizar_constantes(inicio=None):
    """Devuelve la carpeta que contiene constantes.py.

    Busca desde la ubicación del cuaderno hacia arriba. En cada nivel prueba
    la carpeta misma y su subcarpeta 'Bloque A', que es donde vive hoy el
    módulo. Una versión anterior de este cuaderno fijaba la ruta a mano
    (una ruta F: literal), de modo que no corría en otra máquina: es
    exactamente lo que el lineamiento pide evitar.
    """
    inicio = Path(inicio or Path.cwd()).resolve()
    for base in [inicio, *inicio.parents]:
        for candidata in (base, base / "Bloque A"):
            if (candidata / "constantes.py").is_file():
                return candidata
    raise FileNotFoundError(
        "No se encontró constantes.py subiendo desde " + str(inicio)
    )


RUTA_CONSTANTES = localizar_constantes()
if str(RUTA_CONSTANTES) not in sys.path:
    sys.path.insert(0, str(RUTA_CONSTANTES))

import constantes

print("constantes.py cargado desde:", RUTA_CONSTANTES / "constantes.py")
print("RUTA_PROYECTO:            ", constantes.RUTA_PROYECTO)
print("Salidas de este cuaderno: ", constantes.ruta_entregable("B3.1"))

constantes.py cargado desde: F:\Tesis\Bloque A\constantes.py
RUTA_PROYECTO:             F:\Tesis
Salidas de este cuaderno:  F:\Tesis\Bloque B\Entregable B3.1


In [2]:
# =============================================================================
# BLOQUE B3.1 — Evaluación externa del emparejamiento por puntaje combinado
# Celda 2 — Alcance, rutas y parámetros de la etiqueta
# =============================================================================
# ORDEN DEL BLOQUE B (lineal, sin ciclos):
#   B1    empareja por distancia; clasifica; deja 6 casos sin verificar
#   B2    mide la trama urbana y resuelve casos con manzana catastral (UAECD)
#   B2.1  aplica los veredictos y contrasta contra B1
#   B3    reemplaza el emparejamiento por un puntaje combinado
#   B3.1  evalúa ese emparejamiento contra la manzana catastral   <-- aquí
#
# QUÉ HACE ESTE CUADERNO, Y QUÉ NO HACE
#
# Hace: tomar los pares que B3 ya decidió y contrastarlos contra una fuente
# que B3 no usó —la manzana catastral— para medir cuántas de sus decisiones
# quedan respaldadas, cuántas quedan sin respaldo y cuántas quedan
# contradichas.
#
# NO hace: ajustar ningún parámetro de B3. Esto no es una cautela retórica;
# es la corrección del defecto central de la versión anterior de este
# cuaderno. Aquella versión evaluaba el umbral contra estas etiquetas y
# después lo movía de 0,50 a 0,60 «por la evaluación de B3.1», y el propio
# B3_parametros.json lo dejaba escrito. Elegir el parámetro con las mismas
# etiquetas con las que luego se lo evalúa es sobreajuste sobre una muestra
# de una veintena de observaciones, y el número resultante no mide nada.
#
# Hoy el umbral se decide dentro de B3, con un criterio de costo declarado
# antes de mirar estas etiquetas, y este cuaderno lo recibe fijo desde
# B3_parametros.json. Si la evaluación resultara desfavorable, la salida
# correcta NO es mover el umbral hasta que mejore: es declarar el caso y
# resolverlo con evidencia, como hizo B2.
#
# LIMITACIONES DE LA ETIQUETA, DECLARADAS DE ENTRADA
#
#   1. La manzana es un PROXY, no verdad de campo, y es asimétrica:
#         - manzanas distintas  -> evidencia fuerte de sitios distintos
#         - misma manzana       -> compatibilidad, no confirmación
#      Un polígono de varias hectáreas contiene dos estaciones que nada
#      tienen que ver. Por eso más abajo los positivos y los negativos NO se
#      tratan como etiquetas del mismo peso, y la tasa de acierto global se
#      reporta pero no se usa como conclusión.
#
#   2. La capa es de catastro distrital: cubre Bogotá y no Medellín. Los
#      pares candidatos de Medellín quedan fuera de la evaluación. Es la
#      misma limitación ya declarada en B2 y en B3 para la escala de 101,7 m.
#
#   3. El conjunto evaluable es de dos docenas de pares. Permite comprobar
#      concordancia y detectar contradicciones; no permite estimar tasas de
#      acierto con precisión útil. Se reportan conteos absolutos.

RUTA_B1   = constantes.ruta_entregable("B1")
RUTA_B2   = constantes.ruta_entregable("B2")
RUTA_B2_1 = constantes.ruta_entregable("B2.1")
RUTA_B3   = constantes.ruta_entregable("B3")
RUTA_OUT  = Path(constantes.ruta_entregable("B3.1"))
RUTA_OUT.mkdir(parents=True, exist_ok=True)

CRS_GEO  = constantes.CRS_WGS84     # EPSG:4326 — como vienen OSM y OCM
CRS_PROY = constantes.CRS_METRICO   # EPSG:9377 — MAGNA-SIRGAS, métrico

# -----------------------------------------------------------------------------
# Corte de área: cuándo una manzana deja de discriminar
#
# Una manzana urbana normal es un polígono de unos pocos miles de metros
# cuadrados. Los polígonos de cientos de miles corresponden a parques,
# aeropuertos o predios institucionales enteros, y decir que dos estaciones
# «están en la misma manzana» de 114 hectáreas no aporta información.
#
# El valor se declara aquí y en la celda de carga se imprime a qué percentil
# de la capa corresponde, de modo que quede medido y no elegido a ojo.
# -----------------------------------------------------------------------------
AREA_MAX_MZ_M2 = 100_000.0

# -----------------------------------------------------------------------------
# Tolerancia de contención: BANDA, no valor único
#
# Un punto puede caer unos metros fuera del polígono por error posicional de
# la fuente o por cómo la capa traza el lindero. Hay que admitir una
# tolerancia, y ahí está el riesgo: la versión anterior de este cuaderno usó
# 25 m fijos, que es el mismo umbral de confirmación que el método bajo
# evaluación. Validar un criterio con una tolerancia tomada del criterio es
# circular, y B2 ya corrigió exactamente ese defecto: al pasar de 25 m fijos
# a una banda medida sobre la propia capa catastral, sus veredictos adoptados
# bajaron de 4 a 2.
#
# Aquí se hereda la banda de B2 y se aplica su misma regla:
# SOLO SE ADOPTA LA ETIQUETA QUE NO CAMBIA EN NINGÚN VALOR DE LA BANDA.
# Si el veredicto depende de la tolerancia, la etiqueta es «no concluyente».
# -----------------------------------------------------------------------------
TOLERANCIAS_DECLARADAS = [2.7, 6.4, 10.8, 21.5]

print("EVALUACIÓN DE B3 CONTRA LA MANZANA CATASTRAL")
print("-" * 78)
print(f"  Cobertura de la etiqueta:  Bogotá únicamente")
print(f"  Corte de área de manzana:  {AREA_MAX_MZ_M2:,.0f} m²")
print(f"  Ajuste de parámetros aquí: ninguno")
print(f"\n  Salidas en: {RUTA_OUT}")

EVALUACIÓN DE B3 CONTRA LA MANZANA CATASTRAL
------------------------------------------------------------------------------
  Cobertura de la etiqueta:  Bogotá únicamente
  Corte de área de manzana:  100,000 m²
  Ajuste de parámetros aquí: ninguno

  Salidas en: F:\Tesis\Bloque B\Entregable B3.1


# Procedencia de los insumos

In [3]:
# Este cuaderno consume salidas de B3 y la capa cruda que descargó B2. Si
# cualquiera de las dos cambia, la evaluación describe otra cosa. La cadena se
# comprueba contra los manifiestos que dejaron B2 y B3, igual que hacen los
# cuadernos anteriores: un insumo regenerado tiene que fallar con un mensaje,
# no cambiar los números en silencio.

ARCH_MANIFIESTO_B3 = constantes.exigir_archivo(
    constantes.ruta_entregable("B3", "B3_manifiesto_insumos.json"))

with open(ARCH_MANIFIESTO_B3, encoding="utf-8") as fh:
    MANIFIESTO_B3 = json.load(fh)

# --- Insumos propios de B3.1 -------------------------------------------------
INSUMOS = {
    "B3_parametros.json":
        constantes.ruta_entregable("B3", "B3_parametros.json"),
    "B3_pares_puntaje_combinado.csv":
        constantes.ruta_entregable("B3", "B3_pares_puntaje_combinado.csv"),
    "B3_candidatos_evaluados.csv":
        constantes.ruta_entregable("B3", "B3_candidatos_evaluados.csv"),
    "B3_candidatos_rechazados.csv":
        constantes.ruta_entregable("B3", "B3_candidatos_rechazados.csv"),
    "A2_estaciones_OSM_Bogota.geojson":
        constantes.ruta_entregable("A2", "A2_estaciones_OSM_Bogota.geojson"),
    "A2_estaciones_OSM_Medellin.geojson":
        constantes.ruta_entregable("A2", "A2_estaciones_OSM_Medellin.geojson"),
    "A3_OpenChargeMap_Bogota_estaciones.csv":
        constantes.ruta_entregable("A3", "A3_OpenChargeMap_Bogota_estaciones.csv"),
    "A3_OpenChargeMap_Medellin_estaciones.csv":
        constantes.ruta_entregable("A3", "A3_OpenChargeMap_Medellin_estaciones.csv"),
    "B2_veredictos.json":
        constantes.ruta_entregable("B2", "B2_veredictos.json"),
    "B2_1_clasificacion_verificada.csv":
        constantes.ruta_entregable("B2.1", "B2_1_clasificacion_verificada.csv"),
}

HASHES_INSUMOS = {}
for nombre, ruta in INSUMOS.items():
    constantes.exigir_archivo(ruta)
    HASHES_INSUMOS[nombre] = constantes.sha256_archivo(ruta)

print("PROCEDENCIA DE LOS INSUMOS")
print("=" * 78)

discrepancias, nuevos = [], []
for nombre, calculado in HASHES_INSUMOS.items():
    registrado = MANIFIESTO_B3.get(nombre)
    if registrado is None:
        estado = "no comparable (producido por B3 o posterior)"
        nuevos.append(nombre)
    elif registrado == calculado:
        estado = "coincide con B3"
    else:
        estado = "NO COINCIDE con B3"
        discrepancias.append(nombre)
    print(f"  {nombre}")
    print(f"      {calculado[:16]}…  {estado}")

if discrepancias:
    raise RuntimeError(
        "Los insumos cambiaron respecto de los que consumió B3: "
        + ", ".join(discrepancias)
        + ".\nEvaluar los pares de B3 contra datos distintos de los que los "
        "produjeron asignaría cada conclusión a otra estación. Si la "
        "regeneración fue intencional, hay que correr la cadena hacia "
        "adelante: B1, B2, B2.1, B3 y recién después B3.1."
    )

# --- Crudos congelados, heredados por la cadena ------------------------------
CRUDOS_HEREDADOS = {k: v for k, v in MANIFIESTO_B3.items()
                    if k.startswith("crudo::")}
if not CRUDOS_HEREDADOS:
    raise RuntimeError(
        "El manifiesto de B3 no trae entradas 'crudo::'. Sin ellas la cadena "
        "de procedencia no llega a los archivos congelados de A2 y A3."
    )

print(f"\n  Insumos comparados contra B3: {len(HASHES_INSUMOS) - len(nuevos)}")
print(f"  Registrados sin comparar:     {len(nuevos)}")
for n in nuevos:
    print(f"      {n}")
print(f"  Crudos congelados heredados:  {len(CRUDOS_HEREDADOS)}")
for n in sorted(CRUDOS_HEREDADOS):
    print(f"      {n.removeprefix('crudo::')}")
print(f"\n  Fecha de corte del inventario: {constantes.FECHA_CORTE_LOCAL}")

PROCEDENCIA DE LOS INSUMOS
  B3_parametros.json
      20fc689e8ab3c1e4…  no comparable (producido por B3 o posterior)
  B3_pares_puntaje_combinado.csv
      6c288bede04300c0…  no comparable (producido por B3 o posterior)
  B3_candidatos_evaluados.csv
      fcc7a15519931011…  no comparable (producido por B3 o posterior)
  B3_candidatos_rechazados.csv
      10f5c15066d4fedc…  no comparable (producido por B3 o posterior)
  A2_estaciones_OSM_Bogota.geojson
      99f132a15e2a87e9…  coincide con B3
  A2_estaciones_OSM_Medellin.geojson
      69a8c0bafbf86fce…  coincide con B3
  A3_OpenChargeMap_Bogota_estaciones.csv
      cb12e624d5aa1856…  coincide con B3
  A3_OpenChargeMap_Medellin_estaciones.csv
      02767749ab5e1857…  coincide con B3
  B2_veredictos.json
      b9e0d7ee9f101cb3…  coincide con B3
  B2_1_clasificacion_verificada.csv
      b0a2e926cf09b73c…  coincide con B3

  Insumos comparados contra B3: 6
  Registrados sin comparar:     4
      B3_parametros.json
      B3_pares_puntaje_co

# Parámetros y resultados de B3, leídos de disco

In [4]:
# Los parámetros NO se redeclaran aquí con su valor literal. Se leen del
# archivo que produjo B3, de modo que si allá cambian, esta evaluación los
# recoge sin editar nada y no existen dos copias que se desincronicen.
#
# La lectura es por clave explícita y falla con un mensaje si el esquema
# cambió. La versión anterior de este cuaderno leía PAR["umbral_puntaje"] y
# PAR["pares_aceptados"], claves que el B3 actual ya no escribe: el cuaderno
# reventaba en la tercera celda con un KeyError pelado.

ARCH_PAR = constantes.ruta_entregable("B3", "B3_parametros.json")
with open(ARCH_PAR, encoding="utf-8") as fh:
    PAR = json.load(fh)


def exigir_clave(dic, ruta):
    """Lee una clave anidada y explica qué falta si no está."""
    actual, recorrido = dic, []
    for clave in ruta:
        recorrido.append(clave)
        if not isinstance(actual, dict) or clave not in actual:
            raise KeyError(
                f"B3_parametros.json no trae {'.'.join(recorrido)}. "
                "El esquema del archivo cambió en B3 y esta celda hay que "
                "actualizarla: no se debe suponer el valor faltante."
            )
        actual = actual[clave]
    return actual


ESCALA_DISTANCIA_M = exigir_clave(PAR, ["escala_distancia_m"])
PESO_DISTANCIA     = exigir_clave(PAR, ["peso_distancia"])
PESO_NOMBRE        = exigir_clave(PAR, ["peso_nombre"])
BONO_OPERADOR      = exigir_clave(PAR, ["bono_operador"])
VETO_OPERADOR      = exigir_clave(PAR, ["veto_operador"])
RADIO_BUSQUEDA_MAX = exigir_clave(PAR, ["radio_busqueda_m"])

UMBRAL_PUNTAJE     = exigir_clave(PAR, ["decision_umbral", "umbral_adoptado"])
UMBRALES_EVALUADOS = exigir_clave(PAR, ["decision_umbral", "umbrales_evaluados"])
CRITERIO_UMBRAL    = exigir_clave(PAR, ["decision_umbral", "criterio"])
ES_CALIBRACION     = exigir_clave(PAR, ["decision_umbral", "es_calibracion"])
PARES_DECLARADOS   = exigir_clave(PAR, ["decision_umbral", "pares_resultantes"])

# El umbral tiene que venir declarado como NO calibrado. Si algún día B3
# escribiera es_calibracion=true, significaría que allá se ajustó contra
# veredictos externos, y entonces esta evaluación dejaría de ser independiente.
assert ES_CALIBRACION is False, (
    "B3 declara haber calibrado el umbral contra evidencia externa. Si esa "
    "evidencia es la manzana catastral, esta evaluación mide el ajuste contra "
    "sus propias etiquetas y no significa nada. Revisar B3 antes de seguir."
)

# -----------------------------------------------------------------------------
# Pares y candidatos
#
# Los identificadores se leen como texto. Con nulos presentes pandas los
# infiere float y escribe '475409.0', que no une contra '475409': ninguna
# comparación cruzaría y la evaluación saldría vacía sin error. Ya rompió una
# verificación de la tutoría una vez.
# -----------------------------------------------------------------------------
TIPOS_ID = {"ocm_id": "string", "osm_uid": "string"}

PARES_B3 = pd.read_csv(constantes.ruta_entregable(
    "B3", "B3_pares_puntaje_combinado.csv"), encoding="utf-8-sig", dtype=TIPOS_ID)
CANDIDATOS = pd.read_csv(constantes.ruta_entregable(
    "B3", "B3_candidatos_evaluados.csv"), encoding="utf-8-sig", dtype=TIPOS_ID)

for tabla in (PARES_B3, CANDIDATOS):
    for col in ("ocm_id", "osm_uid"):
        tabla[col] = tabla[col].str.replace(r"\.0$", "", regex=True)

# --- Consistencia entre el archivo de parámetros y el de pares ---------------
# Dos archivos que describen la misma corrida pueden quedar desfasados si uno
# se regenera y el otro no. Aquí se comprueba, en vez de suponerse.
assert len(PARES_B3) == PARES_DECLARADOS, (
    f"B3_parametros.json declara {PARES_DECLARADOS} pares y "
    f"B3_pares_puntaje_combinado.csv trae {len(PARES_B3)}. Los dos archivos "
    "vienen de corridas distintas: hay que reejecutar B3 completo."
)
assert (PARES_B3.puntaje >= UMBRAL_PUNTAJE - 1e-9).all(), (
    "Hay pares aceptados por debajo del umbral declarado."
)
assert UMBRAL_PUNTAJE in UMBRALES_EVALUADOS

PARES_SET = set(zip(PARES_B3.ocm_id, PARES_B3.osm_uid))
CANDIDATOS["emparejado_b3"] = [
    (o, s) in PARES_SET for o, s in zip(CANDIDATOS.ocm_id, CANDIDATOS.osm_uid)]

assert int(CANDIDATOS.emparejado_b3.sum()) == len(PARES_B3), (
    "Hay pares aceptados que no figuran entre los candidatos evaluados: las "
    "dos salidas de B3 no describen la misma corrida."
)

# -----------------------------------------------------------------------------
# Coordenadas, recuperadas POR IDENTIFICADOR desde las fuentes congeladas
#
# No se toman de la tabla de pares. Si hubiera un error de unión en B3 —el
# defecto original del Bloque A—, tomarlas de allí lo heredaría y la
# evaluación lo confirmaría en vez de detectarlo.
# -----------------------------------------------------------------------------
osm, ocm = constantes.cargar_fuentes_estaciones()
COORD_OCM = ocm.set_index(ocm.id.astype(str))[["lat", "lon"]]
COORD_OSM = osm.set_index(osm.id.astype(str))[["lat", "lon"]]

_faltan_ocm = sorted(set(CANDIDATOS.ocm_id) - set(COORD_OCM.index))
_faltan_osm = sorted(set(CANDIDATOS.osm_uid) - set(COORD_OSM.index))
assert not _faltan_ocm and not _faltan_osm, (
    f"Identificadores que no existen en las fuentes: OCM {_faltan_ocm}, "
    f"OSM {_faltan_osm}. Un par no puede referirse a una estación inexistente."
)

for lado, fuente in (("ocm", COORD_OCM), ("osm", COORD_OSM)):
    clave = "ocm_id" if lado == "ocm" else "osm_uid"
    CANDIDATOS[f"lat_{lado}"] = fuente.lat.reindex(CANDIDATOS[clave]).values
    CANDIDATOS[f"lon_{lado}"] = fuente.lon.reindex(CANDIDATOS[clave]).values

print("PARÁMETROS DE B3 (leídos, no redeclarados)")
print("-" * 78)
print(f"  Escala de distancia:  {ESCALA_DISTANCIA_M} m")
print(f"  Pesos:                distancia {PESO_DISTANCIA} · nombre {PESO_NOMBRE}")
print(f"  Ajuste por operador:  bono {BONO_OPERADOR:+.2f} · veto {VETO_OPERADOR:+.2f}")
print(f"  Umbral adoptado:      {UMBRAL_PUNTAJE}   (declarado NO calibrado)")
print(f"  Criterio del umbral:  {CRITERIO_UMBRAL}")
print(f"\n  Pares aceptados por B3:  {len(PARES_B3)}")
print(f"  Candidatos evaluados:    {len(CANDIDATOS)}  "
      f"(Bogotá {int((CANDIDATOS.ciudad == 'Bogota').sum())}, "
      f"Medellín {int((CANDIDATOS.ciudad == 'Medellin').sum())})")

PARÁMETROS DE B3 (leídos, no redeclarados)
------------------------------------------------------------------------------
  Escala de distancia:  101.7 m
  Pesos:                distancia 0.6 · nombre 0.4
  Ajuste por operador:  bono +0.10 · veto +0.00
  Umbral adoptado:      0.5   (declarado NO calibrado)
  Criterio del umbral:  costo asimétrico: ante evidencia equivalente se prefiere el umbral más alto, porque un falso positivo elimina una estación del inventario y un falso negativo solo la duplica

  Pares aceptados por B3:  13
  Candidatos evaluados:    27  (Bogotá 24, Medellín 3)


# Reproducción independiente de la asignación de B3

In [5]:
# Antes de evaluar el resultado de B3 hay que comprobar que este cuaderno lo
# está leyendo bien. Se reimplementa la regla —asignación 1:1 codiciosa por
# puntaje descendente— sobre la tabla de candidatos y se comprueba que
# reproduce exactamente los pares aceptados.
#
# No es una tautología: usa los puntajes ya calculados, pero la ASIGNACIÓN se
# vuelve a hacer. Si en B3 hubiera un desalineamiento entre el puntaje y la
# pareja escrita, o un registro tomado dos veces, aquí no cuadraría. Y es lo
# que habilita el barrido de umbrales de más abajo sin tener que reevaluar
# nombres ni distancias con una segunda implementación —que fue justamente
# otro defecto de la versión anterior: reimplementaba la similitud textual con
# una lista de palabras vacías distinta, incluyendo «parqueadero», «centro» y
# «comercial», que en estos nombres sí discriminan. Con eso los puntajes de la
# evaluación no eran los del método evaluado.


def asignar(cand, umbral, columna_puntaje="puntaje"):
    """Asignación 1:1 codiciosa, por ciudad, en orden de puntaje descendente.

    Desempate determinista: puntaje, luego distancia ascendente, luego
    identificadores. Sin el último criterio el resultado dependería del orden
    de las filas del archivo.
    """
    elegidos = set()
    for ciudad in sorted(cand.ciudad.unique()):
        sub = cand[(cand.ciudad == ciudad)
                   & (cand[columna_puntaje] >= umbral - 1e-12)]
        sub = sub.sort_values(
            by=[columna_puntaje, "distancia_m", "ocm_id", "osm_uid"],
            ascending=[False, True, True, True])
        ocm_tomados, osm_tomados = set(), set()
        for fila in sub.itertuples():
            if fila.ocm_id in ocm_tomados or fila.osm_uid in osm_tomados:
                continue
            ocm_tomados.add(fila.ocm_id)
            osm_tomados.add(fila.osm_uid)
            elegidos.add((fila.ocm_id, fila.osm_uid))
    return elegidos


REPRODUCCION = asignar(CANDIDATOS, UMBRAL_PUNTAJE)

REPRODUCE_B3 = REPRODUCCION == PARES_SET
print("REPRODUCCIÓN DE LA ASIGNACIÓN DE B3")
print("-" * 78)
print(f"  Pares de B3:            {len(PARES_SET)}")
print(f"  Pares reproducidos:     {len(REPRODUCCION)}")
print(f"  Coinciden exactamente:  {'SÍ' if REPRODUCE_B3 else 'NO'}")
if not REPRODUCE_B3:
    print(f"    solo en B3:          {sorted(PARES_SET - REPRODUCCION)}")
    print(f"    solo en reproducción:{sorted(REPRODUCCION - PARES_SET)}")

# -----------------------------------------------------------------------------
# Segunda comprobación: la fórmula del puntaje
#
# Se recompone el puntaje a partir de sus componentes guardados y se contrasta
# contra la columna escrita por B3. Si no coincidiera, el barrido de pesos de
# más abajo estaría moviendo una fórmula distinta de la del método.
# -----------------------------------------------------------------------------

def recomponer(fila, peso_distancia, peso_nombre):
    """Puntaje a partir de los componentes que guardó B3.

    La ausencia de nombre reduce al componente espacial, NUNCA penaliza: es la
    regla de B3 y hay que respetarla al recomponer, o el barrido castigaría a
    la fuente peor documentada.
    """
    if pd.isna(fila.s_nombre):
        base = fila.s_distancia
    else:
        base = peso_distancia * fila.s_distancia + peso_nombre * fila.s_nombre
    return float(np.clip(base + fila.ajuste_operador, 0.0, 1.0))


_recompuesto = np.array([recomponer(f, PESO_DISTANCIA, PESO_NOMBRE)
                         for f in CANDIDATOS.itertuples()])
FORMULA_OK = bool(np.allclose(_recompuesto, CANDIDATOS.puntaje.values, atol=1e-9))
print(f"\n  Puntaje recompuesto == puntaje de B3: {'SÍ' if FORMULA_OK else 'NO'}")
if not FORMULA_OK:
    _dif = np.abs(_recompuesto - CANDIDATOS.puntaje.values)
    print(f"    diferencia máxima: {_dif.max():.6f}")

REPRODUCCIÓN DE LA ASIGNACIÓN DE B3
------------------------------------------------------------------------------
  Pares de B3:            13
  Pares reproducidos:     13
  Coinciden exactamente:  SÍ

  Puntaje recompuesto == puntaje de B3: SÍ


# Etiqueta de referencia: manzana catastral

In [6]:
# La capa es la que ya descargó B2. No se vuelve a descargar: una segunda
# descarga podría traer otra versión y la procedencia quedaría registrada en
# B2_procedencia_manzana.json describiendo un archivo distinto del usado.
import glob

_shps = glob.glob(str(Path(RUTA_B2) / "crudo" / "**" / "*.shp"), recursive=True)
if len(_shps) != 1:
    raise RuntimeError(
        f"Se esperaba un único .shp en la carpeta cruda de B2; se hallaron "
        f"{len(_shps)}. Corra B2 primero, o deje un solo archivo."
    )
ARCH_MANZANA = _shps[0]
HASH_MANZANA = constantes.sha256_archivo(ARCH_MANZANA)

# -----------------------------------------------------------------------------
# Banda de tolerancia, heredada de B2
#
# Se intenta leerla de las salidas de B2 para no mantener dos copias. Si no
# aparece, se usa la banda declarada en la celda 2 y se deja constancia de
# que el origen es una copia local: es un dato que el lector debe poder
# distinguir, no algo que deba resolverse en silencio.
# -----------------------------------------------------------------------------

def _recorrer(dato, prefijo=""):
    if isinstance(dato, dict):
        for k, v in dato.items():
            yield from _recorrer(v, f"{prefijo}.{k}" if prefijo else str(k))
    else:
        yield prefijo, dato


def tolerancias_desde_b2():
    for arch in sorted(Path(RUTA_B2).glob("*.json")):
        try:
            with open(arch, encoding="utf-8") as fh:
                dato = json.load(fh)
        except (json.JSONDecodeError, UnicodeDecodeError, OSError):
            continue
        for clave, valor in _recorrer(dato):
            if ("toleran" in clave.lower() and isinstance(valor, list)
                    and len(valor) >= 2
                    and all(isinstance(v, (int, float)) for v in valor)):
                return sorted(float(v) for v in valor), f"{arch.name} · {clave}"
    return None, None


TOLERANCIAS, ORIGEN_TOLERANCIAS = tolerancias_desde_b2()
if TOLERANCIAS is None:
    TOLERANCIAS = sorted(TOLERANCIAS_DECLARADAS)
    ORIGEN_TOLERANCIAS = ("copia declarada en este cuaderno; no se halló la "
                          "banda en las salidas de B2")

# -----------------------------------------------------------------------------
# Carga y reparación de la capa
# -----------------------------------------------------------------------------
_mz = gpd.read_file(ARCH_MANZANA)
_n_invalidas = int((~_mz.is_valid).sum())
if _n_invalidas:
    _mz.loc[~_mz.is_valid, "geometry"] = _mz.loc[~_mz.is_valid, "geometry"].apply(make_valid)
    _mz = _mz[_mz.geom_type.isin(["Polygon", "MultiPolygon"])].copy()

MANZANAS = _mz.to_crs(CRS_PROY)
MANZANAS["area_mz_m2"] = MANZANAS.area

COL_MZ = "MANCODIGO"
if COL_MZ not in MANZANAS.columns:
    raise KeyError(
        f"La capa no trae la columna {COL_MZ}. Columnas disponibles: "
        f"{sorted(MANZANAS.columns)}"
    )

print("CAPA DE MANZANA CATASTRAL (UAECD, descargada por B2)")
print("-" * 78)
print(f"  Archivo:   {Path(ARCH_MANZANA).name}")
print(f"  sha256:    {HASH_MANZANA[:16]}…")
print(f"  Polígonos: {len(MANZANAS):,}  (geometrías reparadas: {_n_invalidas})")

# --- El corte de área, medido sobre la capa ----------------------------------
_pct = float((MANZANAS.area_mz_m2 <= AREA_MAX_MZ_M2).mean() * 100)
print(f"\n  Corte de área declarado: {AREA_MAX_MZ_M2:,.0f} m²")
print(f"    deja por debajo el {_pct:.2f} % de las manzanas de la capa")
print(f"    mediana {MANZANAS.area_mz_m2.median():,.0f} m² · "
      f"p99 {MANZANAS.area_mz_m2.quantile(0.99):,.0f} m²")
print("  No es un valor elegido a ojo: separa la manzana urbana del predio")
print("  institucional o el parque, que es donde la contención deja de decir")
print("  algo sobre si dos puntos son el mismo sitio.")

print(f"\n  Banda de tolerancia: {TOLERANCIAS} m")
print(f"    origen: {ORIGEN_TOLERANCIAS}")
print("  Solo se adopta la etiqueta que no cambia en ningún valor de la banda.")

CAPA DE MANZANA CATASTRAL (UAECD, descargada por B2)
------------------------------------------------------------------------------
  Archivo:   MANZ.shp
  sha256:    8ddaab4fc3872a55…
  Polígonos: 44,518  (geometrías reparadas: 8)

  Corte de área declarado: 100,000 m²
    deja por debajo el 99.45 % de las manzanas de la capa
    mediana 2,508 m² · p99 65,425 m²
  No es un valor elegido a ojo: separa la manzana urbana del predio
  institucional o el parque, que es donde la contención deja de decir
  algo sobre si dos puntos son el mismo sitio.

  Banda de tolerancia: [2.7, 6.4, 10.8, 21.5] m
    origen: B2_auditoria_procesamiento.json · parametros_b2.BANDA_TOLERANCIA_M
  Solo se adopta la etiqueta que no cambia en ningún valor de la banda.


# Ubicación de los puntos y etiquetado

In [7]:
# Un punto por ESTACIÓN, no por par: una estación que aparece en tres pares
# candidatos se ubica una sola vez, y así la ubicación no puede diferir entre
# dos filas que hablan de la misma estación.

EVAL = CANDIDATOS[CANDIDATOS.ciudad == "Bogota"].copy()

_pts = pd.concat([
    EVAL[["ocm_id", "lat_ocm", "lon_ocm"]].rename(
        columns={"ocm_id": "id", "lat_ocm": "lat", "lon_ocm": "lon"}).assign(fuente="OCM"),
    EVAL[["osm_uid", "lat_osm", "lon_osm"]].rename(
        columns={"osm_uid": "id", "lat_osm": "lat", "lon_osm": "lon"}).assign(fuente="OSM"),
], ignore_index=True).drop_duplicates("id").reset_index(drop=True)

PUNTOS = gpd.GeoDataFrame(_pts, geometry=gpd.points_from_xy(_pts.lon, _pts.lat),
                          crs=CRS_GEO).to_crs(CRS_PROY)

# Verificación independiente de la geometría: al deshacer la proyección deben
# reaparecer las coordenadas de origen. Detecta un intercambio de argumentos en
# points_from_xy, que desplazaría todo el inventario sin lanzar ningún error.
_vuelta = PUNTOS.to_crs(CRS_GEO)
assert np.allclose(_vuelta.geometry.x.values, PUNTOS.lon.values, atol=1e-7)
assert np.allclose(_vuelta.geometry.y.values, PUNTOS.lat.values, atol=1e-7)

# -----------------------------------------------------------------------------
# Manzana más cercana a cada punto
#
# El desempate se fija por código de manzana: si un punto equidista de dos
# polígonos, sjoin_nearest devuelve los dos y el orden de llegada decidiría.
# Ordenar antes de quedarse con uno hace el resultado reproducible.
# -----------------------------------------------------------------------------
UBICACION = gpd.sjoin_nearest(
    PUNTOS, MANZANAS[[COL_MZ, "area_mz_m2", "geometry"]],
    how="left", distance_col="dist_mz_m")
UBICACION = (UBICACION.sort_values(["id", "dist_mz_m", COL_MZ])
             .drop_duplicates("id").set_index("id"))

print("UBICACIÓN DE LOS PUNTOS")
print("-" * 78)
print(f"  Estaciones ubicadas: {len(UBICACION)}")
print(f"    dentro de un polígono:            "
      f"{int((UBICACION.dist_mz_m <= 1e-6).sum())}")
for t in TOLERANCIAS:
    n = int(((UBICACION.dist_mz_m > 1e-6) & (UBICACION.dist_mz_m <= t)).sum())
    print(f"    fuera, pero dentro de {t:>5.1f} m:      {n}")
print(f"    a más de {max(TOLERANCIAS):.1f} m de toda manzana: "
      f"{int((UBICACION.dist_mz_m > max(TOLERANCIAS)).sum())}")
print("\n  Los puntos que caen fuera son los que hacen que la tolerancia")
print("  importe, y por eso se evalúan en toda la banda y no en un valor.")


# -----------------------------------------------------------------------------
# Etiquetado
# -----------------------------------------------------------------------------
def etiquetar_con_tolerancia(oid, ouid, tol):
    """Veredicto de la manzana para una tolerancia dada.

    True  = misma manzana, y la manzana es lo bastante pequeña para decir algo
    False = manzanas distintas
    None  = la manzana no discrimina
    """
    a, b = UBICACION.loc[oid], UBICACION.loc[ouid]
    if a.dist_mz_m > tol or b.dist_mz_m > tol:
        return None, f"algún punto a más de {tol:.1f} m de toda manzana"
    if a[COL_MZ] != b[COL_MZ]:
        return False, f"manzanas distintas: {a[COL_MZ]} vs {b[COL_MZ]}"
    if a.area_mz_m2 > AREA_MAX_MZ_M2:
        return None, (f"misma manzana {a[COL_MZ]}, pero de "
                      f"{a.area_mz_m2:,.0f} m²: no discrimina")
    return True, f"misma manzana {a[COL_MZ]} ({a.area_mz_m2:,.0f} m²)"


def etiquetar(oid, ouid):
    """Etiqueta adoptada: solo la que no cambia en toda la banda.

    Es la regla de B2. Un veredicto que aparece a 21,5 m y desaparece a 2,7 m
    no es un veredicto: es una consecuencia de la tolerancia elegida, y
    adoptarlo sería volver al defecto que B2 ya corrigió.
    """
    veredictos = [etiquetar_con_tolerancia(oid, ouid, t) for t in TOLERANCIAS]
    valores = {v for v, _ in veredictos}
    if len(valores) == 1 and None not in valores:
        return veredictos[0][0], veredictos[0][1], True
    if valores == {None}:
        return None, veredictos[-1][1], False
    _detalle = " | ".join(f"{t:.1f} m: {v}" for t, (v, _) in zip(TOLERANCIAS, veredictos))
    return None, f"depende de la tolerancia ({_detalle})", False


_et = [etiquetar(o, s) for o, s in zip(EVAL.ocm_id, EVAL.osm_uid)]
EVAL["etiqueta"] = [e[0] for e in _et]
EVAL["motivo_etiqueta"] = [e[1] for e in _et]
EVAL["estable_en_la_banda"] = [e[2] for e in _et]

# Comparación con lo que habría dado la tolerancia única de 25 m que usaba la
# versión anterior. No para usarla, sino para medir cuánto cambia la
# conclusión al quitar la circularidad: es un resultado del trabajo.
_et25 = [etiquetar_con_tolerancia(o, s, 25.0)[0]
         for o, s in zip(EVAL.ocm_id, EVAL.osm_uid)]
EVAL["etiqueta_tol_25m"] = _et25

N_ETIQUETADOS = int(EVAL.etiqueta.notna().sum())
print("\nETIQUETAS ADOPTADAS")
print("-" * 78)
print(f"  Pares candidatos de Bogotá:  {len(EVAL)}")
print(f"  Con etiqueta adoptada:       {N_ETIQUETADOS}")
print(f"    misma manzana (positiva):     {int((EVAL.etiqueta == True).sum())}")
print(f"    manzanas distintas (negativa):{int((EVAL.etiqueta == False).sum())}")
print(f"  Sin etiqueta:                {len(EVAL) - N_ETIQUETADOS}")
if (EVAL.etiqueta.isna()).any():
    print()
    print(EVAL[EVAL.etiqueta.isna()].motivo_etiqueta.value_counts().to_string())

_n25 = int(pd.Series(_et25).notna().sum())
print(f"\n  Con la tolerancia única de 25 m habría {_n25} etiquetas "
      f"(hoy {N_ETIQUETADOS}).")
print("  La diferencia es el precio de quitar la circularidad: menos")
print("  etiquetas, pero ninguna que dependa del número que se evaluaba.")

UBICACIÓN DE LOS PUNTOS
------------------------------------------------------------------------------
  Estaciones ubicadas: 42
    dentro de un polígono:            35
    fuera, pero dentro de   2.7 m:      1
    fuera, pero dentro de   6.4 m:      4
    fuera, pero dentro de  10.8 m:      6
    fuera, pero dentro de  21.5 m:      7
    a más de 21.5 m de toda manzana: 0

  Los puntos que caen fuera son los que hacen que la tolerancia
  importe, y por eso se evalúan en toda la banda y no en un valor.

ETIQUETAS ADOPTADAS
------------------------------------------------------------------------------
  Pares candidatos de Bogotá:  24
  Con etiqueta adoptada:       12
    misma manzana (positiva):     5
    manzanas distintas (negativa):7
  Sin etiqueta:                12

motivo_etiqueta
depende de la tolerancia (2.7 m: None | 6.4 m: True | 10.8 m: True | 21.5 m: True)      3
misma manzana 005506001, pero de 1,145,791 m²: no discrimina                            3
misma manzana 008415

# Concordancia, leída de forma asimétrica

In [8]:
# LA TABLA DE CONTINGENCIA SE REPORTA, PERO NO SE LEE COMO UNA TASA DE ACIERTO.
#
# Las dos etiquetas no valen lo mismo:
#
#   «manzanas distintas» -> evidencia fuerte. Dos puntos en manzanas
#      catastrales distintas, con la tolerancia entera a favor, no son la misma
#      estación. Si B3 las emparejó, es un ERROR de B3, y de los graves: una
#      fusión indebida borra una estación del inventario.
#
#   «misma manzana» -> compatibilidad, no confirmación. Dice que nada impide
#      que sean el mismo sitio. Que B3 no las empareje NO es necesariamente un
#      error: puede ser prudencia ante evidencia débil.
#
# Por eso el resultado se presenta en dos paneles de distinto peso y la cifra
# que se defiende en la sustentación es la del primero.

E = EVAL[EVAL.etiqueta.notna()].copy()
E["etiqueta"] = E.etiqueta.astype(bool)
PRED = E.emparejado_b3.astype(bool)

CONTRADICCIONES = E[PRED & ~E.etiqueta]                 # empareja lo distinto
RESPALDADOS     = E[PRED & E.etiqueta]                  # empareja lo compatible
SEPARADOS_OK    = E[~PRED & ~E.etiqueta]                # separa lo distinto
NO_SOSTENIDOS   = E[~PRED & E.etiqueta]                 # separa lo compatible

print("=" * 78)
print(f"PANEL 1 · EVIDENCIA FUERTE  —  {len(CONTRADICCIONES) + len(SEPARADOS_OK)} pares "
      f"en manzanas distintas")
print("=" * 78)
print(f"  B3 los separa (correcto):     {len(SEPARADOS_OK)}")
print(f"  B3 los empareja (ERROR):      {len(CONTRADICCIONES)}")
if len(CONTRADICCIONES):
    print("\n  Cada uno de estos pares es una fusión indebida. No se corrige")
    print("  moviendo el umbral: se declara como caso y se resuelve con")
    print("  evidencia, igual que hizo B2 con los suyos.")
    for r in CONTRADICCIONES.itertuples():
        print(f"\n    {r.ocm_id} <-> {r.osm_uid}")
        print(f"      {r.nombre_ocm}  ||  {r.nombre_osm}")
        print(f"      {r.distancia_m:.1f} m · puntaje {r.puntaje:.3f}")
        print(f"      {r.motivo_etiqueta}")
else:
    print("\n  Ninguna. B3 no empareja ningún par que la manzana separa con")
    print("  evidencia estable en toda la banda de tolerancia.")

print("\n" + "=" * 78)
print(f"PANEL 2 · EVIDENCIA DÉBIL  —  {len(RESPALDADOS) + len(NO_SOSTENIDOS)} pares "
      f"en la misma manzana")
print("=" * 78)
print(f"  B3 los empareja (respaldado): {len(RESPALDADOS)}")
print(f"  B3 no los empareja:           {len(NO_SOSTENIDOS)}")
if len(NO_SOSTENIDOS):
    print("\n  No son errores demostrados: la manzana solo dice que podrían")
    print("  ser el mismo sitio. Son los candidatos a revisar si alguna vez")
    print("  se consigue una tercera fuente que los resuelva.")
    for r in NO_SOSTENIDOS.itertuples():
        _falta = UMBRAL_PUNTAJE - r.puntaje
        print(f"\n    {r.ocm_id} <-> {r.osm_uid}")
        print(f"      {r.nombre_ocm}  ||  {r.nombre_osm}")
        print(f"      {r.distancia_m:.1f} m · puntaje {r.puntaje:.3f} "
              f"(le faltan {_falta:.3f} para el umbral)")
        print(f"      {r.motivo_etiqueta}")

# -----------------------------------------------------------------------------
# Tabla de contingencia, para el informe
# -----------------------------------------------------------------------------
VP, FP = len(RESPALDADOS), len(CONTRADICCIONES)
FN, VN = len(NO_SOSTENIDOS), len(SEPARADOS_OK)

print("\n" + "=" * 78)
print(f"TABLA DE CONTINGENCIA  (n = {len(E)} pares con etiqueta adoptada)")
print("=" * 78)
print("                          misma manzana   manzanas distintas")
print(f"  B3 empareja                    {VP:>3}                {FP:>3}")
print(f"  B3 no empareja                 {FN:>3}                {VN:>3}")
print(f"\n  Concordancia: {VP + VN} de {len(E)}")
print("\n  ADVERTENCIA DE LECTURA: esta concordancia NO es una tasa de acierto.")
print("  La columna izquierda no es verdad de campo y la muestra es de dos")
print("  docenas de casos. La cifra que sí tiene fuerza probatoria es la del")
print(f"  panel 1: {FP} contradicciones contra evidencia estable.")

# -----------------------------------------------------------------------------
# Casos que este cuaderno declara para verificación posterior
#
# Mismo mecanismo que usó B1 para dejar sus seis casos: se escriben, no se
# resuelven aquí.
# -----------------------------------------------------------------------------
CASOS = pd.concat([
    CONTRADICCIONES.assign(tipo="contradiccion",
                           prioridad="alta",
                           accion="fusión indebida: verificar en campo o con tercera fuente"),
    NO_SOSTENIDOS.assign(tipo="compatible_no_sostenido",
                         prioridad="media",
                         accion="posible falso negativo: sin evidencia externa que lo resuelva"),
], ignore_index=True)
if len(CASOS):
    CASOS.insert(0, "caso", [f"B31-{i+1}" for i in range(len(CASOS))])
print(f"\n  Casos declarados para verificación posterior: {len(CASOS)}")

PANEL 1 · EVIDENCIA FUERTE  —  7 pares en manzanas distintas
  B3 los separa (correcto):     7
  B3 los empareja (ERROR):      0

  Ninguna. B3 no empareja ningún par que la manzana separa con
  evidencia estable en toda la banda de tolerancia.

PANEL 2 · EVIDENCIA DÉBIL  —  5 pares en la misma manzana
  B3 los empareja (respaldado): 5
  B3 no los empareja:           0

TABLA DE CONTINGENCIA  (n = 12 pares con etiqueta adoptada)
                          misma manzana   manzanas distintas
  B3 empareja                      5                  0
  B3 no empareja                   0                  7

  Concordancia: 12 de 12

  ADVERTENCIA DE LECTURA: esta concordancia NO es una tasa de acierto.
  La columna izquierda no es verdad de campo y la muestra es de dos
  docenas de casos. La cifra que sí tiene fuerza probatoria es la del
  panel 1: 0 contradicciones contra evidencia estable.

  Casos declarados para verificación posterior: 0


# Robustez: descripción, no elección

In [9]:
# ESTA CELDA NO ELIGE NADA. Recorre umbrales y pesos para mostrar cuán
# sensible es la concordancia a los parámetros, que es información útil para
# el informe. El parámetro adoptado ya está fijado en B3 con un criterio
# declarado antes de ver estas etiquetas.
#
# La versión anterior de este cuaderno hacía justamente lo contrario: miraba
# esta tabla y movía el umbral al valor que daba mejor concordancia. Con eso,
# la concordancia reportada dejaba de ser una medida y pasaba a ser el
# resultado de haber optimizado sobre los mismos 19 casos. Aquí la tabla se
# imprime y se guarda; ninguna variable del método cambia después de ella.

_etiquetas = E.set_index(["ocm_id", "osm_uid"]).etiqueta.to_dict()


def concordancia(conjunto_pares):
    """Cuenta contra las etiquetas adoptadas, con la asimetría explícita."""
    vp = fp = fn = vn = 0
    for clave, real in _etiquetas.items():
        pred = clave in conjunto_pares
        if pred and real:
            vp += 1
        elif pred and not real:
            fp += 1
        elif not pred and real:
            fn += 1
        else:
            vn += 1
    return vp, fp, fn, vn


print("=" * 78)
print("SENSIBILIDAD AL UMBRAL")
print("=" * 78)
print(f"{'umbral':>7} {'pares':>6} {'contradicciones':>16} {'respaldados':>12} "
      f"{'concordancia':>13}")

_filas = []
for u in sorted(set(UMBRALES_EVALUADOS) | {UMBRAL_PUNTAJE}):
    _conj = asignar(CANDIDATOS, u)
    vp, fp, fn, vn = concordancia(_conj)
    marca = "  <== adoptado" if abs(u - UMBRAL_PUNTAJE) < 1e-9 else ""
    print(f"{u:>7.2f} {len(_conj):>6} {fp:>16} {vp:>12} "
          f"{vp + vn:>7} / {len(E):<3}{marca}")
    _filas.append({"umbral": u, "pares": len(_conj), "contradicciones": fp,
                   "respaldados": vp, "no_sostenidos": fn,
                   "separados_correctamente": vn, "concordancia": vp + vn,
                   "evaluados": len(E),
                   "adoptado": abs(u - UMBRAL_PUNTAJE) < 1e-9})

SENSIBILIDAD_UMBRAL = pd.DataFrame(_filas)

_sin_contradiccion = SENSIBILIDAD_UMBRAL[SENSIBILIDAD_UMBRAL.contradicciones == 0]
print(f"\n  Umbrales sin ninguna contradicción de evidencia fuerte: "
      f"{list(_sin_contradiccion.umbral)}")
print("  Es lo único que esta tabla permite afirmar. La concordancia total")
print("  mezcla las dos columnas y no distingue un error grave de una")
print("  prudencia razonable.")

# -----------------------------------------------------------------------------
# Sensibilidad a los pesos
# -----------------------------------------------------------------------------
print("\n" + "=" * 78)
print("SENSIBILIDAD AL PESO DEL NOMBRE  (umbral fijo en el adoptado)")
print("=" * 78)
print(f"{'peso nombre':>12} {'pares':>6} {'contradicciones':>16} {'concordancia':>13}")

_filas_peso = []
for pn in [0.0, 0.2, 0.3, 0.4, 0.5, 0.6]:
    _tmp = CANDIDATOS.copy()
    _tmp["puntaje_alt"] = [recomponer(f, 1 - pn, pn) for f in _tmp.itertuples()]
    _conj = asignar(_tmp, UMBRAL_PUNTAJE, columna_puntaje="puntaje_alt")
    vp, fp, fn, vn = concordancia(_conj)
    marca = "  <== adoptado" if abs(pn - PESO_NOMBRE) < 1e-9 else ""
    print(f"{pn:>12.2f} {len(_conj):>6} {fp:>16} {vp + vn:>7} / {len(E):<3}{marca}")
    _filas_peso.append({"peso_nombre": pn, "peso_distancia": 1 - pn,
                        "pares": len(_conj), "contradicciones": fp,
                        "respaldados": vp, "concordancia": vp + vn,
                        "evaluados": len(E),
                        "adoptado": abs(pn - PESO_NOMBRE) < 1e-9})

SENSIBILIDAD_PESO = pd.DataFrame(_filas_peso)

print("\n  Si la concordancia apenas se mueve al variar el peso, la conclusión")
print("  no es que el peso esté bien elegido: es que el componente textual")
print("  decide poco en este inventario, y hay que decirlo así.")

SENSIBILIDAD AL UMBRAL
 umbral  pares  contradicciones  respaldados  concordancia
   0.40     15                0            5      12 / 12 
   0.45     15                0            5      12 / 12 
   0.50     13                0            5      12 / 12   <== adoptado
   0.55     12                0            5      12 / 12 
   0.60     11                0            5      12 / 12 
   0.65     10                0            5      12 / 12 
   0.70      8                0            3      10 / 12 

  Umbrales sin ninguna contradicción de evidencia fuerte: [0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7]
  Es lo único que esta tabla permite afirmar. La concordancia total
  mezcla las dos columnas y no distingue un error grave de una
  prudencia razonable.

SENSIBILIDAD AL PESO DEL NOMBRE  (umbral fijo en el adoptado)
 peso nombre  pares  contradicciones  concordancia
        0.00     12                0      12 / 12 
        0.20     12                0      12 / 12 
        0.30     12    

# Pruebas de aceptación y guardado

In [10]:
PRUEBAS = []


def prueba(nombre, condicion, detalle=""):
    PRUEBAS.append({"prueba": nombre, "pasa": bool(condicion), "detalle": detalle})


# 1. El cuaderno lee bien lo que evalúa.
prueba("la asignación de B3 se reproduce desde los candidatos",
       REPRODUCE_B3,
       f"{len(REPRODUCCION & PARES_SET)} comunes de {len(PARES_SET)}")

prueba("el puntaje se recompone desde sus componentes",
       FORMULA_OK, "coincide con la columna escrita por B3")

# 2. Cada par evaluado se ubicó, y se ubicó una sola vez.
prueba("cada estación de los pares evaluados tiene una única ubicación",
       UBICACION.index.is_unique
       and set(EVAL.ocm_id) <= set(UBICACION.index)
       and set(EVAL.osm_uid) <= set(UBICACION.index),
       f"{len(UBICACION)} estaciones ubicadas")

# 3. La etiqueta adoptada es estable en toda la banda. Es la prueba que
#    impide volver al defecto circular de la tolerancia única.
_inestables_adoptadas = EVAL[EVAL.etiqueta.notna() & ~EVAL.estable_en_la_banda]
prueba("ninguna etiqueta adoptada depende de la tolerancia",
       len(_inestables_adoptadas) == 0,
       f"banda {TOLERANCIAS} m")

# 4. Ningún parámetro del método se tocó en este cuaderno.
_par_ahora = {
    "escala_distancia_m": ESCALA_DISTANCIA_M,
    "peso_distancia": PESO_DISTANCIA,
    "peso_nombre": PESO_NOMBRE,
    "bono_operador": BONO_OPERADOR,
    "veto_operador": VETO_OPERADOR,
    "radio_busqueda_m": RADIO_BUSQUEDA_MAX,
}
_par_archivo = {k: PAR[k] for k in _par_ahora}
_umbral_archivo = PAR["decision_umbral"]["umbral_adoptado"]
prueba("los parámetros siguen siendo los de B3 al terminar el cuaderno",
       _par_ahora == _par_archivo and UMBRAL_PUNTAJE == _umbral_archivo,
       "ninguna celda posterior a la lectura los modifica")

# 5. Coherencia con lo que B2 ya había resuelto. Los dos pares que B2 descartó
#    con la manzana tienen que salir aquí como manzanas distintas, si es que
#    aparecen entre los candidatos: es la misma capa y la misma regla.
VERIF = pd.read_csv(constantes.ruta_entregable(
    "B2.1", "B2_1_clasificacion_verificada.csv"), encoding="utf-8-sig",
    dtype={"ocm_id": "string", "osm_uid": "string"})
for col in ("ocm_id", "osm_uid"):
    VERIF[col] = VERIF[col].str.replace(r"\.0$", "", regex=True)

_desc = VERIF[(VERIF.verificacion_externa == "descartado") & VERIF.caso.notna()]
PARES_PROHIBIDOS = set()
for caso in sorted(_desc.caso.dropna().unique()):
    filas = _desc[_desc.caso == caso]
    for o in filas.ocm_id.dropna().unique():
        for s in filas.osm_uid.dropna().unique():
            PARES_PROHIBIDOS.add((o, s))

_prohibidos_evaluados = EVAL[[(o, s) in PARES_PROHIBIDOS
                              for o, s in zip(EVAL.ocm_id, EVAL.osm_uid)]]
prueba("los pares que B2 descartó se etiquetan aquí como manzanas distintas",
       bool((_prohibidos_evaluados.etiqueta == False).all())
       if len(_prohibidos_evaluados) else True,
       f"{len(_prohibidos_evaluados)} de {len(PARES_PROHIBIDOS)} presentes "
       "entre los candidatos de Bogotá")

prueba("B3 no empareja ninguno de los pares que B2 descartó",
       len(PARES_SET & PARES_PROHIBIDOS) == 0,
       f"{len(PARES_PROHIBIDOS)} pares prohibidos")

# 6. Cobertura declarada: Medellín queda fuera y tiene que decirse con un
#    número, no con una frase.
_medellin = int((CANDIDATOS.ciudad == "Medellin").sum())
prueba("los candidatos de Medellín quedan fuera de la evaluación, contados",
       int((EVAL.ciudad == "Medellin").sum()) == 0,
       f"{_medellin} pares candidatos sin etiqueta posible")

# 7. Los insumos no cambiaron mientras corría el cuaderno.
for nombre in ("B3_pares_puntaje_combinado.csv", "B3_parametros.json"):
    prueba(f"{nombre} no fue modificado durante la ejecución",
           constantes.sha256_archivo(INSUMOS[nombre]) == HASHES_INSUMOS[nombre])

RESULTADO_PRUEBAS = pd.DataFrame(PRUEBAS)

print("=" * 78)
print("PRUEBAS DE ACEPTACIÓN")
print("=" * 78)
for p in PRUEBAS:
    print(f"  [{'OK  ' if p['pasa'] else 'FALLA'}] {p['prueba']}")
    if p["detalle"]:
        print(f"           {p['detalle']}")
print(f"\n  {int(RESULTADO_PRUEBAS.pasa.sum())} de {len(PRUEBAS)} pasan")

# -----------------------------------------------------------------------------
# Guardado
#
# Sin marca de tiempo de ejecución, deliberadamente: una hora dentro del
# archivo cambia su hash en cada corrida y vuelve el hash inútil como control.
# -----------------------------------------------------------------------------
COLUMNAS_EVAL = ["ciudad", "ocm_id", "osm_uid", "nombre_ocm", "nombre_osm",
                 "distancia_m", "puntaje", "s_distancia", "s_nombre",
                 "ajuste_operador", "emparejado_b3", "etiqueta",
                 "estable_en_la_banda", "motivo_etiqueta", "etiqueta_tol_25m"]

SALIDAS = {
    "B3_1_conjunto_evaluacion.csv": EVAL[COLUMNAS_EVAL],
    "B3_1_contradicciones.csv": CONTRADICCIONES[COLUMNAS_EVAL],
    "B3_1_compatibles_no_sostenidos.csv": NO_SOSTENIDOS[COLUMNAS_EVAL],
    "B3_1_casos_declarados.csv": CASOS,
    "B3_1_sensibilidad_umbral.csv": SENSIBILIDAD_UMBRAL,
    "B3_1_sensibilidad_peso.csv": SENSIBILIDAD_PESO,
    "B3_1_ubicacion_manzana.csv": UBICACION.reset_index()[
        ["id", "fuente", "lat", "lon", COL_MZ, "area_mz_m2", "dist_mz_m"]],
    "B3_1_pruebas_aceptacion.csv": RESULTADO_PRUEBAS,
}

print("\n" + "=" * 78)
print("GUARDADO")
print("=" * 78)
for nombre, tabla in SALIDAS.items():
    tabla.to_csv(RUTA_OUT / nombre, index=False, encoding="utf-8-sig")
    print(f"  {nombre:<44} {len(tabla):>4} filas")

RESUMEN = {
    "bloque": "B3.1",
    "proposito": ("evaluación externa del emparejamiento de B3 contra la "
                  "contención en manzana catastral UAECD"),
    "ajuste_de_parametros": "ninguno; los parámetros se leen de B3 y no se tocan",
    "fecha_corte_inventario": constantes.FECHA_CORTE_LOCAL,
    "cobertura": {
        "ciudad_evaluada": "Bogota",
        "motivo": ("la capa de manzana es de catastro distrital y no cubre "
                   "Medellín"),
        "candidatos_sin_evaluar": _medellin,
    },
    "etiqueta": {
        "fuente": Path(ARCH_MANZANA).name,
        "sha256": HASH_MANZANA,
        "banda_de_tolerancia_m": TOLERANCIAS,
        "origen_banda": ORIGEN_TOLERANCIAS,
        "corte_area_m2": AREA_MAX_MZ_M2,
        "regla": ("solo se adopta la etiqueta que no cambia en ningún valor "
                  "de la banda"),
        "asimetria": ("manzanas distintas es evidencia fuerte; misma manzana "
                      "es compatibilidad, no confirmación"),
    },
    "parametros_evaluados": PAR,
    "resultados": {
        "candidatos_bogota": int(len(EVAL)),
        "con_etiqueta_adoptada": int(len(E)),
        "con_etiqueta_a_25m_fijos": int(pd.Series(_et25).notna().sum()),
        "contradicciones_evidencia_fuerte": int(FP),
        "separados_correctamente": int(VN),
        "respaldados": int(VP),
        "compatibles_no_sostenidos": int(FN),
        "concordancia": int(VP + VN),
    },
    "limitaciones_declaradas": [
        "la etiqueta de manzana es un proxy y es asimétrica: «manzanas "
        "distintas» prueba diferencia, «misma manzana» solo prueba "
        "compatibilidad",
        "la capa cubre únicamente Bogotá; los candidatos de Medellín no se "
        "evalúan",
        "el conjunto etiquetado es de dos docenas de pares: permite detectar "
        "contradicciones, no estimar tasas de acierto",
        "la concordancia global no debe leerse como exactitud del método",
    ],
    "pruebas_pasan": bool(RESULTADO_PRUEBAS.pasa.all()),
}

with open(RUTA_OUT / "B3_1_resumen.json", "w", encoding="utf-8") as fh:
    json.dump(RESUMEN, fh, ensure_ascii=False, indent=2)

MANIFIESTO = dict(HASHES_INSUMOS)
MANIFIESTO["manzana::" + Path(ARCH_MANZANA).name] = HASH_MANZANA
MANIFIESTO.update(CRUDOS_HEREDADOS)
with open(RUTA_OUT / "B3_1_manifiesto_insumos.json", "w", encoding="utf-8") as fh:
    json.dump(MANIFIESTO, fh, ensure_ascii=False, indent=2, sort_keys=True)

# Relectura de ida y vuelta: lo escrito tiene que volver a leerse igual.
_rele = pd.read_csv(RUTA_OUT / "B3_1_conjunto_evaluacion.csv",
                    encoding="utf-8-sig", dtype={"ocm_id": "string",
                                                 "osm_uid": "string"})
_ok_relectura = len(_rele) == len(EVAL) and set(_rele.columns) == set(COLUMNAS_EVAL)
print(f"\n  B3_1_resumen.json y B3_1_manifiesto_insumos.json escritos")
print(f"  Relectura de ida y vuelta: {'OK' if _ok_relectura else 'FALLA'}")
print(f"\n  Archivos en: {RUTA_OUT}")
print(f"  Pruebas de aceptación: "
      f"{'TODAS PASAN' if RESULTADO_PRUEBAS.pasa.all() else 'HAY FALLAS'}")
print("\n  Las salidas de B1, B2, B2.1 y B3 quedan intactas.")

PRUEBAS DE ACEPTACIÓN
  [OK  ] la asignación de B3 se reproduce desde los candidatos
           13 comunes de 13
  [OK  ] el puntaje se recompone desde sus componentes
           coincide con la columna escrita por B3
  [OK  ] cada estación de los pares evaluados tiene una única ubicación
           42 estaciones ubicadas
  [OK  ] ninguna etiqueta adoptada depende de la tolerancia
           banda [2.7, 6.4, 10.8, 21.5] m
  [OK  ] los parámetros siguen siendo los de B3 al terminar el cuaderno
           ninguna celda posterior a la lectura los modifica
  [OK  ] los pares que B2 descartó se etiquetan aquí como manzanas distintas
           2 de 2 presentes entre los candidatos de Bogotá
  [OK  ] B3 no empareja ninguno de los pares que B2 descartó
           2 pares prohibidos
  [OK  ] los candidatos de Medellín quedan fuera de la evaluación, contados
           3 pares candidatos sin etiqueta posible
  [OK  ] B3_pares_puntaje_combinado.csv no fue modificado durante la ejecución
  [OK  ]

# Qué concluye B3.1

**Este cuaderno evalúa; no ajusta.** Los parámetros de B3 se leen de
`B3_parametros.json` y ninguna celda los modifica. Una prueba de aceptación lo
comprueba al final, porque la versión anterior de este cuaderno sí los movía:
miraba la tabla de concordancia y subía el umbral de 0,50 a 0,60 «por la
evaluación de B3.1». Elegir un parámetro con las mismas etiquetas con las que
después se lo evalúa no mide nada, y el propio archivo de parámetros dejaba
escrita la circularidad.

## Las dos etiquetas no pesan igual

| etiqueta | qué prueba | qué significa que B3 discrepe |
|---|---|---|
| manzanas distintas | los puntos están en polígonos catastrales distintos en toda la banda de tolerancia: son sitios distintos | error grave: fusión indebida, borra una estación del inventario |
| misma manzana | nada impide que sean el mismo sitio | no es un error demostrado: puede ser prudencia ante evidencia débil |

Por eso el resultado se lee en el panel 1 —contradicciones contra evidencia
fuerte— y no en la concordancia global. La concordancia se reporta porque el
informe la pide, con la advertencia de que mezcla las dos columnas.

## Corrección heredada de B2: la tolerancia es una banda

La versión anterior admitía puntos hasta 25 m fuera del polígono. Son los
mismos 25 m del umbral de confirmación del método bajo evaluación, de modo que
la etiqueta se apoyaba en el número que estaba juzgando. B2 ya había corregido
ese defecto en su propio cuaderno —sus veredictos adoptados bajaron de 4 a 2 al
hacerlo— y aquí se aplica la misma regla: **solo se adopta la etiqueta que no
cambia en ningún valor de la banda**.

El cuaderno imprime cuántas etiquetas se habrían obtenido con los 25 m fijos.
La diferencia no es una pérdida: es el número que antes se estaba afirmando sin
respaldo.

## Qué hacer con lo que salga

- **Contradicciones (panel 1).** Se declaran como casos en
  `B3_1_casos_declarados.csv` y se resuelven con evidencia, igual que B1 declaró
  sus seis casos y B2 resolvió dos. No se corrigen moviendo el umbral: mover el
  umbral hasta que la contradicción desaparezca es, otra vez, ajustar contra la
  etiqueta.
- **Compatibles no sostenidos (panel 2).** Quedan registrados como posibles
  falsos negativos sin evidencia que los resuelva. Son los primeros candidatos
  si aparece una tercera fuente.

## Limitaciones que van al informe

1. La capa de manzana cubre Bogotá y no Medellín; los candidatos de Medellín
   quedan sin evaluar y el cuaderno los cuenta.
2. El conjunto etiquetado es de dos docenas de pares: sirve para detectar
   contradicciones, no para estimar tasas de acierto.
3. «Misma manzana» depende del tamaño del polígono. El corte de área está
   declarado y el cuaderno imprime a qué percentil de la capa corresponde.